# 04 · Answer Quality Evaluation (LLM-as-judge)

Runs the full RAG pipeline (hybrid retrieval + Gemini answer with citations) on the evaluation set and scores each answer.

| Metric | What it measures |
|---|---|
| **Correctness (1–5)** | Does the answer match the reference answer, including exact numbers? |
| **Faithfulness (1–5)** | Is every claim supported by the retrieved context (no hallucination)? |
| **Citation rate** | Share of answers that cite at least one passage |
| **Refusal accuracy** | Unanswerable questions answered with "I could not find this" (and answerable ones *not* refused) |

> Requires `GEMINI_API_KEY`. About 190 API calls are needed. On the free tier, use a Flash-Lite model (`GEMINI_CHAT_MODEL` / `GEMINI_JUDGE_MODEL` in `.env`). The loop saves progress after every question, so if the daily quota runs out, re-run it later and it resumes.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
import time
from src import config
from src.retriever import build_retriever
from src.generator import GeminiGenerator, RAGPipeline
from src.evaluation import judge_answer
from src.gemini_rest import GeminiError
assert config.GEMINI_API_KEY, "Set GEMINI_API_KEY in .env first"
eval_df = pd.read_csv(config.EVAL_SET_PATH)
rag = RAGPipeline(build_retriever("hybrid"), GeminiGenerator())
judge = GeminiGenerator(model=config.JUDGE_MODEL)
print("Answer model:", config.CHAT_MODEL, "| Judge model:", config.JUDGE_MODEL, "| Retriever:", rag.retriever.name)

Answer model: gemini-3.5-flash-lite | Judge model: gemini-3.5-flash-lite | Retriever: hybrid


In [ ]:
PAUSE_SECONDS = 6  # pause between API calls to stay under the free-tier requests-per-minute limit

In [ ]:
# Resumable loop: results are saved after every question, so if the free-tier
# quota runs out you can simply re-run this cell later and it continues where it stopped.
RESULTS_PATH = Path("reports/rag_eval_results.csv")
done = pd.read_csv(RESULTS_PATH) if RESULTS_PATH.exists() else pd.DataFrame()
done_ids = set(done["qid"]) if len(done) else set()
todo = eval_df[~eval_df["qid"].isin(done_ids)]
print(f"Already evaluated: {len(done_ids)} | remaining: {len(todo)}")

for q in todo.itertuples():
    try:
        ans = rag.ask(q.question)
        row = {"qid": q.qid, "question_type": q.question_type, "question": q.question,
               "answer": ans.text, "refused": ans.is_refusal, "n_citations": len(ans.cited)}
        if q.question_type != "unanswerable":
            time.sleep(PAUSE_SECONDS)
            row.update(judge_answer(judge, ans, q.reference_answer))
    except GeminiError as exc:
        print(f"Stopped at {q.qid}: {str(exc)[:200]}")
        print("Progress is saved. Re-run this cell later (the free quota resets daily) to continue.")
        break
    pd.DataFrame([row]).to_csv(RESULTS_PATH, mode="a", header=not RESULTS_PATH.exists(), index=False)
    print(f"{q.qid} done")
    time.sleep(PAUSE_SECONDS)

res = pd.read_csv(RESULTS_PATH)
print(f"Evaluated {len(res)} of {len(eval_df)} questions")
res.tail()

Already evaluated: 66 | remaining: 0
Evaluated 66 of 66 questions


,qid,question_type,question,answer,refused,n_citations,correctness,faithfulness,reason
61,Q062,unanswerable,What is the firm's policy on hiring interns?,I could not find this in the knowledge base.,True,0,NaN,NaN,NaN
62,Q063,unanswerable,What is the official DJKI fee for a trademark application in 2026?,I could not find this in the knowledge base.,True,0,NaN,NaN,NaN
63,Q064,unanswerable,Who is the current Managing Partner of the firm?,I could not find this in the knowledge base.,True,0,NaN,NaN,NaN
64,Q065,unanswerable,What is our policy on geographical indications?,I could not find this in the knowledge base.,True,0,NaN,NaN,NaN
65,Q066,unanswerable,How much is the firm's annual revenue?,I could not find this in the knowledge base.,True,0,NaN,NaN,NaN


In [ ]:
ans_df = res[res.question_type != "unanswerable"]
una_df = res[res.question_type == "unanswerable"]
scorecard = pd.Series({
    "correctness (avg 1-5)": ans_df["correctness"].mean(),
    "faithfulness (avg 1-5)": ans_df["faithfulness"].mean(),
    "fully correct answers (score 5)": (ans_df["correctness"] == 5).mean(),
    "answers with citation": (ans_df["n_citations"] > 0).mean(),
    "false refusals (answerable but refused)": ans_df["refused"].mean(),
    "correct refusals (unanswerable)": una_df["refused"].mean(),
}).round(3)
scorecard

correctness (avg 1-5)                      4.836
faithfulness (avg 1-5)                     5.000
fully correct answers (score 5)            0.885
answers with citation                      0.902
false refusals (answerable but refused)    0.016
correct refusals (unanswerable)            1.000
dtype: float64

In [ ]:
ans_df.groupby("question_type")[["correctness", "faithfulness"]].mean().round(2)

,correctness,faithfulness
question_type,,
factual,4.86,5.0
paraphrase,4.79,5.0


In [ ]:
# Lowest-scoring answers for error analysis
ans_df.sort_values(["correctness", "faithfulness"]).head(8)[["qid", "question", "answer", "correctness", "reason"]]

,qid,question,answer,correctness,reason
14,Q015,A competitor's similar mark was just published. By when do we need the clien...,I could not find this in the knowledge base.,1.0,"The assistant stated the information was not found, but the answer is indeed..."
9,Q010,How many days do we have to respond to a provisional refusal from the examiner?,The official deadline to respond to a provisional refusal is **30 days** fro...,4.0,"The answer correctly gives the 30-day official deadline from the context, th..."
15,Q016,What is the internal target for drafting a counter-statement when our client...,The firm's internal target is to deliver the draft counter-statement to the ...,4.0,"The answer is fully faithful to the context and mostly correct, though it om..."
17,Q018,How long does an Indonesian trademark registration last?,"An Indonesian trademark registration lasts **10 years** [1, 5].",4.0,"The answer correctly states the 10-year duration, though it omits the renewa..."
18,Q019,At which points before expiry do clients get renewal reminders?,"Clients receive trademark renewal reminders at **12 months, 6 months, 3 mont...",4.0,The assistant mentions the general renewal reminders but also includes extra...
44,Q045,Which currency are foreign clients invoiced in?,Foreign clients are invoiced in USD [1].,4.0,The assistant correctly states the currency but omits the exchange rate deta...
59,Q060,What does a cease-and-desist letter cost?,"A cease-and-desist letter has a fixed professional fee of **IDR 5,000,000** ...",4.0,"The answer provides the correct fee mentioned in the context, though it omit..."
0,Q001,How much does a knock-out trademark search cost per class?,"A knock-out search costs IDR 750,000 per class [1].",5.0,The answer accurately provides the correct fee per class based on the context.


### Business impact framing
Use the scorecard to report results in business terms, for example:
- *x% of staff questions answered fully correctly with a source citation, in seconds instead of minutes of searching.*
- *Unanswerable questions are refused rather than guessed, reducing the risk of a wrong deadline or fee being given to a client.*

Record the actual numbers from your run in the README results table.